# Naive Bayes Baseline: Causal Relation Extraction (3-class)

**Dataset:** CausalNewsCorpus  
Train: `train_clean.txt` | Test: `dev_clean.txt` 


In [ ]:
# Setup
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..'))

TRAIN_PATH  = '../data/CausalNewsCorpus/train_clean.txt'
TEST_PATH   = '../data/CausalNewsCorpus/dev_clean.txt'
RESULTS_DIR = '../results/naive_bayes/cnc/'

from src.shared.data_loader import load_cnc_data, get_label_distribution
from src.shared.evaluation import evaluate

from src.naive_bayes.features import extract_local_context, create_vectorizer
from src.naive_bayes.naive_bayes_model import build_naive_bayes_pipeline

print('Setup complete.')

Setup complete.


In [ ]:
# Data Exploration
train_examples = load_cnc_data(TRAIN_PATH)
print(f'Training examples: {len(train_examples)}')

print('\n=== CLASS DISTRIBUTION (training) ===')
for label, count, pct in get_label_distribution(train_examples):
    print(f'  {label:<26}  {count:>5}  ({pct:.1f}%)')

Training examples: 2023

=== CLASS DISTRIBUTION (training) ===
  Cause-Effect(e2,e1)          1266  (62.6%)
  Other                         399  (19.7%)
  Cause-Effect(e1,e2)           358  (17.7%)


In [ ]:
# Feature Extraction and Training
# Features: 2-word local context window around each entity
train_features = [extract_local_context(ex['sentence'], window=2)
                  for ex in train_examples]
train_labels   = [ex['label'] for ex in train_examples]

print(f'Feature string example: "{train_features[0]}"')
print(f'Label: {train_labels[0]}')

# Build pipeline: CountVectorizer + MultinomialNB (alpha=1.0)
vectorizer = create_vectorizer()
model = build_naive_bayes_pipeline(vectorizer)

print('\nTraining...')
model.fit(train_features, train_labels)
print('Done.')
print(f"Vocabulary size: {len(model.named_steps['bow'].vocabulary_)} tokens")

Feature string example: "e1_l:despite e1:assurances e1:by e1:the e1:authorities e1_r:, e1_r:the e2_l:authorities e2_l:, e2:the e2:stock e2:of e2:urea e2:is e2:not e2:reaching e2:farmers"
Label: Cause-Effect(e1,e2)

Training...
Done.
Vocabulary size: 11659 tokens


In [ ]:
# Predict and Evaluate
test_examples = load_cnc_data(TEST_PATH)
print(f'Test examples: {len(test_examples)}')

print('\n=== CLASS DISTRIBUTION (test) ===')
for label, count, pct in get_label_distribution(test_examples):
    print(f'  {label:<26}  {count:>5}  ({pct:.1f}%)')

test_features = [extract_local_context(ex['sentence'], window=2)
                 for ex in test_examples]
y_true = [ex['label'] for ex in test_examples]
y_pred = model.predict(test_features).tolist()

# Save raw predictions
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, 'predictions.txt'), 'w', encoding='utf-8') as f:
    for label in y_pred:
        f.write(label + '\n')

# Evaluate with shared framework
metrics = evaluate(
    y_true       = y_true,
    y_pred       = y_pred,
    model_name   = 'naive_bayes',
    dataset_name = 'cnc',
    output_dir   = RESULTS_DIR,
)

Test examples: 219

=== CLASS DISTRIBUTION (test) ===
  Cause-Effect(e2,e1)           152  (69.4%)
  Other                          34  (15.5%)
  Cause-Effect(e1,e2)            33  (15.1%)

  EVALUATION: naive_bayes  |  dataset: cnc

  *** PRIMARY METRIC ***
  Macro F1 (Cause-Effect classes only): 0.6948  (69.48%)

  Per-class results:
  Label                            P       R      F1   Support
  --------------------------------------------------------
  Cause-Effect(e1,e2)         0.6842  0.3939  0.5000        33
  Cause-Effect(e2,e1)         0.8142  0.9803  0.8896       152
  Other                       0.9412  0.4706  0.6275        34

  Overall:
    Macro F1 (all 3 classes): 0.6723
    Micro F1:                 0.8128
    Accuracy:                 0.8128


Results saved to: ../results/naive_bayes/cnc/
  report.txt
  metrics.json
  confusion_matrix.png
